# NodeMapper tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/angelphanth/hugging-mapper/blob/main/docs/tutorial/nodemapper-tutorial.ipynb)


In [ ]:
# uncomment if colab
#!pip install pandas hugging-mapper

Returning node ids based on similarity of text embeddings.

Start by importing `NodeMapper`

In [1]:
from hugger.mapper import NodeMapper

Demo data for the tutorial

In [2]:
# An example dataframe
import pandas as pd

# generate data
ids = ["id1", "id2", "id3", "id4", "id5", "id6", "id7", "id8"]
texts = [
    "They are happy",
    "I would like to order a doughnut",
    "The grass is green",
    "They are sad",
    "Have you poured the foundation?",
    "I am feeling grey",
    "blue",
    "home",
]
# to dataframe
df = pd.DataFrame({"id": ids, "text": texts})

Initializing `NodeMapper` will 
- load the given huggingface model
- generate embeddings for the text column
- creating a dictionary of the node ids : text embeddings

In [3]:
# init
mapper = NodeMapper(
    df=df,
    text_col="text",
    id_col="id",
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating embeddings for 8 nodes ...


Like `HuggingMapper` can simply get embeddings for given text

In [4]:
# generate embedding for a single text
embedding = mapper.embed_text("Good morning")
print(embedding.shape)

# generate embeddings for a list of texts
embeddings = mapper.embed_text(["Hello world", "Good evening", "Lunch time!"])
print(embeddings.shape)

torch.Size([1, 384])
torch.Size([3, 384])


But the main purpose of `NodeMapper` is to find similar texts and their corresponding ids

In [5]:
# retrieve those most similar to given text, above threshold
mapper.get_similar("concrete", threshold=0)  # threshold 0 returns all

{'id5': {'text': 'Have you poured the foundation?',
  'score': 0.4041473865509033},
 'id8': {'text': 'home', 'score': 0.2964901328086853},
 'id7': {'text': 'blue', 'score': 0.27229541540145874},
 'id3': {'text': 'The grass is green', 'score': 0.19606007635593414},
 'id6': {'text': 'I am feeling grey', 'score': 0.16133837401866913},
 'id2': {'text': 'I would like to order a doughnut',
  'score': 0.13787642121315002},
 'id1': {'text': 'They are happy', 'score': 0.12416310608386993},
 'id4': {'text': 'They are sad', 'score': 0.11256911605596542}}

In [6]:
# retrieve top match, above threshold
print(mapper.get_match("joyful", threshold=0.4), "\n")
print(mapper.get_match("we are crying", threshold=0.4), "\n")
print(mapper.get_match("eatting a donut", threshold=0.4), "\n")

('id1', {'text': 'They are happy', 'score': 0.5418258905410767}) 

('id4', {'text': 'They are sad', 'score': 0.5035914182662964}) 

('id2', {'text': 'I would like to order a doughnut', 'score': 0.5166527032852173}) 



In [7]:
# retrieve top k matches, above threshold
print(mapper.get_similar("yellow", threshold=0.3, top_k=2), "\n")
print(mapper.get_similar("laughter", top_k=3), "\n")

{'id7': {'text': 'blue', 'score': 0.6618931293487549}, 'id6': {'text': 'I am feeling grey', 'score': 0.35393187403678894}} 

{'id1': {'text': 'They are happy', 'score': 0.42021429538726807}, 'id4': {'text': 'They are sad', 'score': 0.36576396226882935}, 'id8': {'text': 'home', 'score': 0.342716783285141}} 

